# Unsupervised Learning: All Methods on One Dataset

This notebook fits 10 unsupervised learning methods (clustering + dimensionality reduction) on compact sklearn datasets and visualizes them side-by-side. See how each method reveals different structure.

**Run All:** Runtime -> Run all (or Ctrl+F9).

Links: [Data Science BSc Course](https://github.com/Digital-AI-Finance/data-science) | [Dashboard](https://digital-ai-finance.github.io/data-science/)

## What is unsupervised learning?

Unsupervised learning finds structure in data **without labels**. Two families:

- **Clustering** groups similar points (K-Means, Hierarchical, DBSCAN, GMM). Output: a cluster label per point.
- **Dimensionality reduction** projects high-dimensional data to 2-3 dimensions for visualization (PCA, t-SNE, Isomap). Output: 2D coordinates per point.

This notebook shows both families on one figure. Row 1 = clustering on 2D blobs. Row 2 = dim-reduction on 8D digit images projected to 2D.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, load_digits
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.manifold import TSNE, Isomap
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [ ]:
NAVY, AMBER = '#1e3a5f', '#d97706'

# Clustering dataset: 4 blobs in 2D
X_blobs, y_blobs_true = make_blobs(n_samples=300, centers=4, cluster_std=1.2, random_state=42)

# Dim-reduction dataset: 8x8 handwritten digits (6 classes kept for speed)
digits = load_digits(n_class=6)
X_digits = StandardScaler().fit_transform(digits.data[:500])  # subsample to 500 for t-SNE speed
y_digits = digits.target[:500]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs_true, cmap='tab10', s=20, edgecolor='k', linewidth=0.3)
ax1.set_title(f'Clustering dataset: {X_blobs.shape[0]} points, 4 true blobs')
ax1.set_xticks([]); ax1.set_yticks([])

ax2.imshow(digits.images[0], cmap='gray')
ax2.set_title(f'Dim-reduction dataset: {X_digits.shape[0]} digits (shape {digits.images[0].shape})')
ax2.set_xticks([]); ax2.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
clustering_models = {
    'K-Means (k=3)':    KMeans(n_clusters=3, n_init=10, random_state=42),
    'K-Means (k=6)':    KMeans(n_clusters=6, n_init=10, random_state=42),
    'Hierarchical Ward':AgglomerativeClustering(n_clusters=4, linkage='ward'),
    'DBSCAN (eps=0.8)': DBSCAN(eps=0.8, min_samples=5),
    'GMM (4 comp)':     GaussianMixture(n_components=4, random_state=42),
}

dimred_models = {
    'PCA':              PCA(n_components=2, random_state=42),
    'Truncated SVD':    TruncatedSVD(n_components=2, random_state=42),
    't-SNE (p=30)':     TSNE(n_components=2, perplexity=30, random_state=42, max_iter=500),
    't-SNE (p=5)':      TSNE(n_components=2, perplexity=5, random_state=42, max_iter=500),
    'Isomap (k=10)':    Isomap(n_components=2, n_neighbors=10),
}

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

# Row 1: Clustering on blobs
for ax, (name, model) in zip(axes[0], clustering_models.items()):
    if hasattr(model, 'fit_predict'):
        labels = model.fit_predict(X_blobs)
    else:
        model.fit(X_blobs)
        labels = model.predict(X_blobs)
    # Silhouette only valid with >=2 unique clusters
    unique_labels = set(labels) - {-1}
    sil = silhouette_score(X_blobs, labels) if len(unique_labels) >= 2 else float('nan')
    ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap='tab10', s=20, edgecolor='k', linewidth=0.3)
    k_found = len(unique_labels)
    sil_str = f'{sil:.2f}' if not np.isnan(sil) else 'n/a'
    ax.set_title(f'{name}\nk={k_found}, silhouette={sil_str}', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

# Row 2: Dim reduction on digits
for ax, (name, model) in zip(axes[1], dimred_models.items()):
    t0 = time.time()
    if hasattr(model, 'fit_transform'):
        Z = model.fit_transform(X_digits)
    else:
        model.fit(X_digits)
        Z = model.transform(X_digits)
    t1 = time.time()
    ax.scatter(Z[:, 0], Z[:, 1], c=y_digits, cmap='tab10', s=15, edgecolor='k', linewidth=0.2)
    ax.set_title(f'{name}\ntime={t1-t0:.1f}s', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('Unsupervised Learning: Row 1 = Clustering on blobs | Row 2 = Dim-reduction on digits (colored by true class)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for name, model in clustering_models.items():
    if hasattr(model, 'fit_predict'):
        labels = model.fit_predict(X_blobs)
    else:
        model.fit(X_blobs)
        labels = model.predict(X_blobs)
    unique_labels = set(labels) - {-1}
    sil = silhouette_score(X_blobs, labels) if len(unique_labels) >= 2 else np.nan
    n_noise = (labels == -1).sum() if -1 in labels else 0
    rows.append({'Family': 'Clustering', 'Method': name, 'Clusters': len(unique_labels),
                 'Noise pts': n_noise, 'Silhouette': f'{sil:.3f}' if not np.isnan(sil) else 'n/a'})

for name, model in dimred_models.items():
    rows.append({'Family': 'Dim-Reduction', 'Method': name, 'Clusters': '-', 'Noise pts': '-', 'Silhouette': '-'})

pd.DataFrame(rows)

## Your Turn: Parameter Exercise

Each unsupervised method has hyperparameters that dramatically change the output. Try these experiments by calling `compare(**kwargs)`:

**Experiment 1 - K-Means wrong k:** What if k=2 on a 4-blob dataset?
```python
compare(kmeans_k=2)
```

**Experiment 2 - DBSCAN eps sensitivity:** Small eps -> many noise points. Large eps -> one giant cluster.
```python
compare(eps=0.3)
compare(eps=2.0)
```

**Experiment 3 - Linkage method:** Ward assumes roughly spherical clusters. Single linkage can chain.
```python
compare(linkage='single')
```

**Experiment 4 - t-SNE perplexity extremes:** Low = local structure. High = global structure.
```python
compare(tsne_perplexity=3)
compare(tsne_perplexity=100)
```

**Experiment 5 - Harder data:** Overlapping blobs break clustering.
```python
compare(blobs_std=2.5)
```

Predict the silhouette score BEFORE running. Then compare.

In [ ]:
def compare(kmeans_k=6, eps=0.8, linkage='ward', gmm_n=4,
            tsne_perplexity=30, blobs_std=1.2, n_samples=300):
    """Rerun all 10 UL methods with custom hyperparameters.

    Parameters
    ----------
    kmeans_k : int, default=6
        Number of clusters for K-Means (row 1 col 2).
    eps : float, default=0.8
        DBSCAN neighborhood radius.
    linkage : str, default='ward'
        Agglomerative linkage: ward, complete, average, single.
    gmm_n : int, default=4
        GMM component count.
    tsne_perplexity : float, default=30
        t-SNE perplexity (effective neighbors).
    blobs_std : float, default=1.2
        Noise level of clustering dataset.
    n_samples : int, default=300
        Clustering dataset size.

    Returns
    -------
    pandas.DataFrame with cluster counts + silhouette scores.

    Examples
    --------
    >>> compare(kmeans_k=2)       # K-Means underfits
    >>> compare(eps=0.3)          # DBSCAN: many noise points
    >>> compare(linkage='single') # Chaining effect
    """
    Xb, _ = make_blobs(n_samples=n_samples, centers=4, cluster_std=blobs_std, random_state=42)

    cm = {
        'K-Means (k=3)':    KMeans(n_clusters=3, n_init=10, random_state=42),
        f'K-Means (k={kmeans_k})': KMeans(n_clusters=kmeans_k, n_init=10, random_state=42),
        f'Hierarchical ({linkage})': AgglomerativeClustering(n_clusters=4, linkage=linkage),
        f'DBSCAN (eps={eps})': DBSCAN(eps=eps, min_samples=5),
        f'GMM ({gmm_n} comp)': GaussianMixture(n_components=gmm_n, random_state=42),
    }
    dm = {
        'PCA':              PCA(n_components=2, random_state=42),
        'Truncated SVD':    TruncatedSVD(n_components=2, random_state=42),
        f't-SNE (p={tsne_perplexity})': TSNE(n_components=2, perplexity=tsne_perplexity, random_state=42, max_iter=500),
        't-SNE (p=5)':      TSNE(n_components=2, perplexity=5, random_state=42, max_iter=500),
        'Isomap (k=10)':    Isomap(n_components=2, n_neighbors=10),
    }

    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    rows = []
    for ax, (name, model) in zip(axes[0], cm.items()):
        if hasattr(model, 'fit_predict'):
            labels = model.fit_predict(Xb)
        else:
            model.fit(Xb)
            labels = model.predict(Xb)
        uniq = set(labels) - {-1}
        sil = silhouette_score(Xb, labels) if len(uniq) >= 2 else float('nan')
        ax.scatter(Xb[:, 0], Xb[:, 1], c=labels, cmap='tab10', s=20, edgecolor='k', linewidth=0.3)
        sil_str = f'{sil:.2f}' if not np.isnan(sil) else 'n/a'
        ax.set_title(f'{name}\nk={len(uniq)}, sil={sil_str}', fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
        rows.append({'Family':'Clust', 'Method':name, 'k':len(uniq), 'Silhouette':sil_str})

    for ax, (name, model) in zip(axes[1], dm.items()):
        t0 = time.time()
        if hasattr(model, 'fit_transform'):
            Z = model.fit_transform(X_digits)
        else:
            model.fit(X_digits)
            Z = model.transform(X_digits)
        ax.scatter(Z[:, 0], Z[:, 1], c=y_digits, cmap='tab10', s=15, edgecolor='k', linewidth=0.2)
        ax.set_title(f'{name}\ntime={time.time()-t0:.1f}s', fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
        rows.append({'Family':'DimRed', 'Method':name, 'k':'-', 'Silhouette':'-'})

    fig.suptitle(f'compare(kmeans_k={kmeans_k}, eps={eps}, linkage={linkage!r}, tsne_perplexity={tsne_perplexity}, blobs_std={blobs_std})', fontsize=10)
    plt.tight_layout()
    plt.show()
    return pd.DataFrame(rows)

compare(kmeans_k=2)  # Example: underfitting

## Expected Results

| Experiment | Call | What Changes | Lesson |
|---|---|---|---|
| Baseline | (default) | K-Means k=6 oversplits; DBSCAN finds 4; t-SNE p=30 separates digits | Default hyperparameters work ok |
| K-Means underfit | `compare(kmeans_k=2)` | Two blobs merged; silhouette drops | Wrong k hurts clustering |
| DBSCAN tight | `compare(eps=0.3)` | Many noise points (label=-1) | eps controls density threshold |
| DBSCAN loose | `compare(eps=2.0)` | One giant cluster | eps too large erases structure |
| Single linkage | `compare(linkage='single')` | Chaining: outliers create long tendrils | Ward is more robust |
| t-SNE tight | `compare(tsne_perplexity=3)` | Digits fragment into sub-groups | Low perplexity = local focus |
| t-SNE loose | `compare(tsne_perplexity=100)` | Classes merge into one blob | High perplexity = global focus |
| Noisy data | `compare(blobs_std=2.5)` | All clustering silhouettes drop | Bayes error limit |

### Big picture

**Unsupervised learning is harder than it looks because there are no labels to tell you if you got the right answer.** You use silhouette score (for clustering) or trustworthiness (for dim-reduction) as proxies for quality. Always sanity-check visually - the 2x5 grid IS the validation.

Next step: use silhouette or BIC scores in a `GridSearchCV`-style sweep to pick hyperparameters automatically.

## Key takeaways

- **Clustering methods** (row 1) draw region boundaries - each color is one cluster. Different methods find different numbers of clusters.
- **Dimensionality reduction** (row 2) projects high-dim data to 2D so you can see structure. Points colored by TRUE class to judge whether the projection preserves it.
- **No labels = no accuracy**. Use silhouette score (higher = tighter clusters) and trustworthiness (how well the projection preserves neighborhoods).
- **Hyperparameters matter as much as algorithm choice** - wrong k or wrong perplexity can hide real structure.

## Next steps

Go deeper on any method with the per-lesson notebooks:
- [L29 K-Means Clustering](https://github.com/Digital-AI-Finance/data-science/tree/master/L29_KMeans_Clustering)
- [L30 Hierarchical Clustering](https://colab.research.google.com/github/Digital-AI-Finance/data-science-notebooks/blob/main/L30_Hierarchical_Clustering/colab/L30_Hierarchical_Clustering_Notebook.ipynb)
- [L31 PCA](https://github.com/Digital-AI-Finance/data-science/tree/master/L31_PCA)
- [L32 ML Pipeline](https://colab.research.google.com/github/Digital-AI-Finance/data-science-notebooks/blob/main/L32_ML_Pipeline/colab/L32_ML_Pipeline_Notebook.ipynb)

Read the [Unsupervised Learning Handout](https://digital-ai-finance.github.io/data-science/teaching_materials/Unsupervised_Learning_Handout.pdf) for complete derivations.

Try the [Quiz](https://digital-ai-finance.github.io/data-science/quiz/unsupervised-learning.html) when ready.